In [1]:
!pwd

/content


In [2]:
import sys;
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [3]:
# !pip install uv

# !uv pip install \
#   "darts==0.34.0" \
#   "torch==2.3.1" \
#   "pytorch-lightning==2.2.5" \
#   "torchmetrics==1.3.2" \
#   "numpy==1.26.4" \
#   "pandas==2.2.2" \
#   "scikit-learn==1.7.2" \
#   "dask==2025.5.0" \
#   "xgboost==3.0.2" \
#   "catboost==1.2.8" \
#   "pyyaml==6.0.2" \
#   "lightgbm==4.6.0"

# Pin torch for reproducibility with your local run
# !uv pip install -q --index-url https://download.pytorch.org/whl/cu121 "torch==2.3.1"

In [1]:
import sys, torch, darts, pytorch_lightning as pl, numpy as np, pandas as pd
print(sys.version)
print("torch:", torch.__version__)
print("darts:", darts.__version__)
print("lightning:", pl.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("cuda:", torch.cuda.is_available())

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch: 2.3.1+cu121
darts: 0.34.0
lightning: 2.2.5
numpy: 1.26.4
pandas: 2.2.2
cuda: True


**Forecasts**
 - **Country: Canada**
 - **Forecast Horizons:12M and 24M Forecast**

In [2]:
import random
import pandas as pd
import numpy as np
import torch

from darts import TimeSeries, concatenate
from darts.dataprocessing.transformers import Scaler
from darts.models import NBEATSModel, TFTModel, TiDEModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict

class canadaForecastGenerator:
  def __init__(self, data_path: str, random_seed: int = 1):
      # Seeds
      random.seed(random_seed)
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      # Stronger deterministic behavior on CPU
      torch.use_deterministic_algorithms(True)

      # Load data
      self.data = pd.read_csv(data_path)

      # Define target and exogenous variables
      self.target_vars = [
          "CPIinflationrate",
          "Unemploymentrate",
          "OilpriceGlobalWTI",
          "RealbroadEER",
          "ShorttermIR"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
      # Split data based on forecast horizon
      train = self.data.head(-forecast_horizon).copy()

      # Create target TimeSeries
      target_series = []
      for var in self.target_vars:
          target_series.append(TimeSeries.from_series(train[var]))

      # Stack target variables
      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      # Create exogenous TimeSeries
      exog_series = []
      for var in self.exog_vars:
          exog_series.append(TimeSeries.from_series(train[var]))

      # Stack exogenous variables
      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def create_model(self, forecast_horizon: int) -> TiDEModel:
      # Keep minimal HPs exactly as requested + force CPU for reproducibility
      return TiDEModel(
          input_chunk_length=6,
          output_chunk_length=forecast_horizon,
          n_epochs=300,
          random_state=0,
          pl_trainer_kwargs={"accelerator": "cpu", "devices": 1}
      )

  def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          # Prepare data
          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          # Create and train model
          model = self.create_model(horizon)
          print(f"Training model for {horizon}-month horizon...")
          model.fit(
              series=train_target_ts,
              past_covariates=train_exog_ts,
              verbose=True
          )

          # Generate forecast
          print(f"Generating {horizon}-month predictions...")
          pred = model.predict(
              n=horizon,
              series=train_target_ts,
              past_covariates=train_exog_ts
          )

          # Keep same conversion style as your original code
          pred_df = pd.DataFrame({
              'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
              'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
              'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
              'forecast_eer': pred.pd_dataframe().iloc[:, 3],
              'forecast_ir': pred.pd_dataframe().iloc[:, 4]
          })

          forecasts[horizon] = pred_df

          # Save forecast to CSV
          output_file = f'canada_forecasts_tidex_{horizon}m.csv'
          pred_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

      return forecasts

def main():
  generator = canadaForecastGenerator(
      data_path='all_mulvar_data_canada_v2.csv',
      random_seed=1
  )

  forecasts = generator.generate_forecasts([12, 24])

  print("\nCreated files:")
  for horizon in [12, 24]:
      print(f"canada_forecasts_tidex_{horizon}m.csv")

if __name__ == "__main__":
  main()


Generating 12-month forecast...


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Training model for 12-month horizon...


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/setup.py:187: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type             | Params
---------------------------------------------------------
0 | criterion           | MSELoss          | 0     
1 | train_criterion     | MSELoss          | 0     
2 | val_criterion       | MSELoss          | 0     
3 | train_metrics       | MetricCollection | 0     
4 | val_metrics         | MetricCollection | 0     
5 | past_cov_projection | _ResidualBlock   | 1.2 K 
6 | encoders            | Sequential       | 30.6 K
7 | decoders            | Sequential       | 66.0 K
8 | temporal_decoder    | _ResidualBlock   | 794   
9 | lookback_skip       | Linear           | 84    
---------------------------------------------------------
98.7 K    Trainable params
0         Non-trainable params
98.7 K    Total params
0.395     Total estim

Training: |          | 0/? [00:00<?, ?it/s]

...

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Generating 24-month predictions...


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast saved to canada_forecasts_tidex_24m.csv

Created files:
canada_forecasts_tidex_12m.csv
canada_forecasts_tidex_24m.csv


**Forecasts**
 - **Country: USA**
 - **Forecast Horizons:12M and 24M Forecast**

In [3]:
import random
import pandas as pd
import numpy as np
import torch

from darts import TimeSeries, concatenate
from darts.dataprocessing.transformers import Scaler
from darts.models import NBEATSModel, TFTModel, TiDEModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict


class usaForecastGenerator:
    def __init__(self, data_path: str, random_seed: int = 1):
        # Seeds for reproducibility
        random.seed(random_seed)
        torch.manual_seed(random_seed)
        np.random.seed(random_seed)
        torch.use_deterministic_algorithms(True)

        # Load data
        self.data = pd.read_csv(data_path)

        # Define target and exogenous variables
        self.target_vars = [
            "CPIinflationrate",
            "Unemploymentrate",
            "OilpriceGlobalWTI",
            "RealbroadEER",
            "ShorttermIR"
        ]

        self.exog_vars = [
            "logEPU",
            "GPRC",
            "USEMV",
            "USMPU"
        ]

    def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
        # Split data based on forecast horizon
        train = self.data.head(-forecast_horizon).copy()

        # Create target TimeSeries
        target_series = []
        for var in self.target_vars:
            target_series.append(TimeSeries.from_series(train[var]))

        # Stack target variables
        train_target_ts = target_series[0]
        for series in target_series[1:]:
            train_target_ts = train_target_ts.stack(series)

        # Create exogenous TimeSeries
        exog_series = []
        for var in self.exog_vars:
            exog_series.append(TimeSeries.from_series(train[var]))

        # Stack exogenous variables
        train_exog_ts = exog_series[0]
        for series in exog_series[1:]:
            train_exog_ts = train_exog_ts.stack(series)

        return train_target_ts, train_exog_ts

    def create_model(self, forecast_horizon: int) -> TiDEModel:
        # Keep minimal HPs + force CPU for reproducibility
        return TiDEModel(
            input_chunk_length=6,
            output_chunk_length=forecast_horizon,
            n_epochs=300,
            random_state=0,
            pl_trainer_kwargs={"accelerator": "cpu", "devices": 1}
        )

    def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
        forecasts = {}

        for horizon in forecast_horizons:
            print(f"\nGenerating {horizon}-month forecast...")

            # Prepare data
            train_target_ts, train_exog_ts = self.prepare_data(horizon)

            # Create and train model
            model = self.create_model(horizon)
            print(f"Training model for {horizon}-month horizon...")
            model.fit(
                series=train_target_ts,
                past_covariates=train_exog_ts,
                verbose=True
            )

            # Generate forecast
            print(f"Generating {horizon}-month predictions...")
            pred = model.predict(
                n=horizon,
                series=train_target_ts,
                past_covariates=train_exog_ts
            )

            # Convert to DataFrame (same style as your existing code)
            pred_df = pd.DataFrame({
                'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
                'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
                'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
                'forecast_eer': pred.pd_dataframe().iloc[:, 3],
                'forecast_ir': pred.pd_dataframe().iloc[:, 4]
            })

            forecasts[horizon] = pred_df

            # Save forecast to CSV
            output_file = f'usa_forecasts_tidex_{horizon}m.csv'
            pred_df.to_csv(output_file, index=True)
            print(f"Forecast saved to {output_file}")

        return forecasts


def main():
    generator = usaForecastGenerator(
        data_path='/content/all_mulvar_data_usa_v2.csv',
        random_seed=1
    )

    forecasts = generator.generate_forecasts([12, 24])

    print("\nCreated files:")
    for horizon in [12, 24]:
        print(f"usa_forecasts_tidex_{horizon}m.csv")


if __name__ == "__main__":
    main()

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/setup.py:187: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type             | Params
---------------------------------------------------------
0 | criterion           | MSELoss          | 0     
1 | train_criterion     | MSELoss          | 0     
2 | val_criterion       | MSELoss          | 0     
3 | train_metrics       | MetricCollection | 0     
4 | val_metrics         | MetricCollection | 0     
5 | past_cov_projection | _ResidualBlock   | 1.2 K 
6 | encoders            | Sequential       


Generating 12-month forecast...
Training model for 12-month horizon...


Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Generating 12-month predictions...


...

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Generating 24-month predictions...


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast saved to usa_forecasts_tidex_24m.csv

Created files:
usa_forecasts_tidex_12m.csv
usa_forecasts_tidex_24m.csv


**Forecasts**
 - **Country: France**
 - **Forecast Horizons:12M and 24M Forecast**

In [5]:
import random
import pandas as pd
import numpy as np
import torch

from darts import TimeSeries, concatenate
from darts.dataprocessing.transformers import Scaler
from darts.models import NBEATSModel, TFTModel, TiDEModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict


class franceForecastGenerator:
    def __init__(self, data_path: str, random_seed: int = 1):
        # Seeds for reproducibility
        random.seed(random_seed)
        torch.manual_seed(random_seed)
        np.random.seed(random_seed)
        torch.use_deterministic_algorithms(True)

        # Load data
        self.data = pd.read_csv(data_path)

        # Define target and exogenous variables
        self.target_vars = [
            "CPIinflationrate",
            "Unemploymentrate",
            "OilpriceGlobalWTI",
            "RealbroadEER",
            "ShorttermIR"
        ]

        self.exog_vars = [
            "logEPU",
            "GPRC",
            "USEMV",
            "USMPU"
        ]

    def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
        # Split data based on forecast horizon
        train = self.data.head(-forecast_horizon).copy()

        # Create target TimeSeries
        target_series = []
        for var in self.target_vars:
            target_series.append(TimeSeries.from_series(train[var]))

        # Stack target variables
        train_target_ts = target_series[0]
        for series in target_series[1:]:
            train_target_ts = train_target_ts.stack(series)

        # Create exogenous TimeSeries
        exog_series = []
        for var in self.exog_vars:
            exog_series.append(TimeSeries.from_series(train[var]))

        # Stack exogenous variables
        train_exog_ts = exog_series[0]
        for series in exog_series[1:]:
            train_exog_ts = train_exog_ts.stack(series)

        return train_target_ts, train_exog_ts

    def create_model(self, forecast_horizon: int) -> TiDEModel:
        # Keep minimal HPs + force CPU for reproducibility
        return TiDEModel(
            input_chunk_length=6,
            output_chunk_length=forecast_horizon,
            n_epochs=300,
            random_state=0,
            pl_trainer_kwargs={"accelerator": "cpu", "devices": 1}
        )

    def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
        forecasts = {}

        for horizon in forecast_horizons:
            print(f"\nGenerating {horizon}-month forecast...")

            # Prepare data
            train_target_ts, train_exog_ts = self.prepare_data(horizon)

            # Create and train model
            model = self.create_model(horizon)
            print(f"Training model for {horizon}-month horizon...")
            model.fit(
                series=train_target_ts,
                past_covariates=train_exog_ts,
                verbose=True
            )

            # Generate forecast
            print(f"Generating {horizon}-month predictions...")
            pred = model.predict(
                n=horizon,
                series=train_target_ts,
                past_covariates=train_exog_ts
            )

            # Convert to DataFrame
            pred_df = pd.DataFrame({
                'forecast_inflation': pred.pd_dataframe().iloc[:, 0],
                'forecast_unemployment': pred.pd_dataframe().iloc[:, 1],
                'forecast_oil_price': pred.pd_dataframe().iloc[:, 2],
                'forecast_eer': pred.pd_dataframe().iloc[:, 3],
                'forecast_ir': pred.pd_dataframe().iloc[:, 4]
            })

            forecasts[horizon] = pred_df

            # Save forecast to CSV
            output_file = f'france_forecasts_tidex_{horizon}m.csv'
            pred_df.to_csv(output_file, index=True)
            print(f"Forecast saved to {output_file}")

        return forecasts


def main():
    generator = franceForecastGenerator(
        data_path='/content/all_mulvar_data_france_v2.csv',
        random_seed=1
    )

    forecasts = generator.generate_forecasts([12, 24])

    print("\nCreated files:")
    for horizon in [12, 24]:
        print(f"france_forecasts_tidex_{horizon}m.csv")


if __name__ == "__main__":
    main()

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/setup.py:187: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type             | Params
---------------------------------------------------------
0 | criterion           | MSELoss          | 0     
1 | train_criterion     | MSELoss          | 0     
2 | val_criterion       | MSELoss          | 0     
3 | train_metrics       | MetricCollection | 0     
4 | val_metrics         | MetricCollection | 0     
5 | past_cov_projection | _ResidualBlock   | 1.2 K 
6 | encoders            | Sequential       


Generating 12-month forecast...
Training model for 12-month horizon...


Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Generating 12-month predictions...


...

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Generating 24-month predictions...


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast saved to france_forecasts_tidex_24m.csv

Created files:
france_forecasts_tidex_12m.csv
france_forecasts_tidex_24m.csv


**Forecasts**
 - **Country: Germany**
 - **Forecast Horizons:12M and 24M Forecast**

In [6]:
import random
import pandas as pd
import numpy as np
import torch

from darts import TimeSeries, concatenate
from darts.dataprocessing.transformers import Scaler
from darts.models import NBEATSModel, TFTModel, TiDEModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict


class germanyForecastGenerator:
    def __init__(self, data_path: str, random_seed: int = 1):
        # Seeds for reproducibility
        random.seed(random_seed)
        torch.manual_seed(random_seed)
        np.random.seed(random_seed)
        torch.use_deterministic_algorithms(True)

        # Load data
        self.data = pd.read_csv(data_path)

        # Define target and exogenous variables
        self.target_vars = [
            "CPIinflationrate",
            "Unemploymentrate",
            "OilpriceGlobalWTI",
            "RealbroadEER",
            "ShorttermIR"
        ]

        self.exog_vars = [
            "logEPU",
            "GPRC",
            "USEMV",
            "USMPU"
        ]

    def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
        # Split data based on forecast horizon
        train = self.data.head(-forecast_horizon).copy()

        # Create target TimeSeries
        target_series = []
        for var in self.target_vars:
            target_series.append(TimeSeries.from_series(train[var]))

        # Stack target variables
        train_target_ts = target_series[0]
        for series in target_series[1:]:
            train_target_ts = train_target_ts.stack(series)

        # Create exogenous TimeSeries
        exog_series = []
        for var in self.exog_vars:
            exog_series.append(TimeSeries.from_series(train[var]))

        # Stack exogenous variables
        train_exog_ts = exog_series[0]
        for series in exog_series[1:]:
            train_exog_ts = train_exog_ts.stack(series)

        return train_target_ts, train_exog_ts

    def create_model(self, forecast_horizon: int) -> TiDEModel:
        # Keep minimal HPs + force CPU for reproducibility
        return TiDEModel(
            input_chunk_length=6,
            output_chunk_length=forecast_horizon,
            n_epochs=300,
            random_state=0,
            pl_trainer_kwargs={"accelerator": "cpu", "devices": 1}
        )

    def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
        forecasts = {}

        for horizon in forecast_horizons:
            print(f"\nGenerating {horizon}-month forecast...")

            # Prepare data
            train_target_ts, train_exog_ts = self.prepare_data(horizon)

            # Create and train model
            model = self.create_model(horizon)
            print(f"Training model for {horizon}-month horizon...")
            model.fit(
                series=train_target_ts,
                past_covariates=train_exog_ts,
                verbose=True
            )

            # Generate forecast
            print(f"Generating {horizon}-month predictions...")
            pred = model.predict(
                n=horizon,
                series=train_target_ts,
                past_covariates=train_exog_ts
            )

            # Convert to DataFrame
            pred_df = pd.DataFrame({
                "forecast_inflation": pred.pd_dataframe().iloc[:, 0],
                "forecast_unemployment": pred.pd_dataframe().iloc[:, 1],
                "forecast_oil_price": pred.pd_dataframe().iloc[:, 2],
                "forecast_eer": pred.pd_dataframe().iloc[:, 3],
                "forecast_ir": pred.pd_dataframe().iloc[:, 4]
            })

            forecasts[horizon] = pred_df

            # Save forecast to CSV
            output_file = f"germany_forecasts_tidex_{horizon}m.csv"
            pred_df.to_csv(output_file, index=True)
            print(f"Forecast saved to {output_file}")

        return forecasts


def main():
    generator = germanyForecastGenerator(
        data_path="/content/all_mulvar_data_germany_v2.csv",
        random_seed=1
    )

    generator.generate_forecasts([12, 24])

    print("\nCreated files:")
    for horizon in [12, 24]:
        print(f"germany_forecasts_tidex_{horizon}m.csv")


if __name__ == "__main__":
    main()

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/setup.py:187: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type             | Params
---------------------------------------------------------
0 | criterion           | MSELoss          | 0     
1 | train_criterion     | MSELoss          | 0     
2 | val_criterion       | MSELoss          | 0     
3 | train_metrics       | MetricCollection | 0     
4 | val_metrics         | MetricCollection | 0     
5 | past_cov_projection | _ResidualBlock   | 1.2 K 
6 | encoders            | Sequential       


Generating 12-month forecast...
Training model for 12-month horizon...


Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Generating 12-month predictions...


...

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Generating 24-month predictions...


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast saved to germany_forecasts_tidex_24m.csv

Created files:
germany_forecasts_tidex_12m.csv
germany_forecasts_tidex_24m.csv


**Forecasts**
 - **Country: Japan**
 - **Forecast Horizons:12M and 24M Forecast**

In [7]:
import random
import pandas as pd
import numpy as np
import torch

from darts import TimeSeries, concatenate
from darts.dataprocessing.transformers import Scaler
from darts.models import NBEATSModel, TFTModel, TiDEModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict


class japanForecastGenerator:
    def __init__(self, data_path: str, random_seed: int = 1):
        # Seeds for reproducibility
        random.seed(random_seed)
        torch.manual_seed(random_seed)
        np.random.seed(random_seed)
        torch.use_deterministic_algorithms(True)

        # Load data
        self.data = pd.read_csv(data_path)

        # Define target and exogenous variables
        self.target_vars = [
            "CPIinflationrate",
            "Unemploymentrate",
            "OilpriceGlobalWTI",
            "RealbroadEER",
            "ShorttermIR"
        ]

        self.exog_vars = [
            "logEPU",
            "GPRC",
            "USEMV",
            "USMPU"
        ]

    def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
        # Split data based on forecast horizon
        train = self.data.head(-forecast_horizon).copy()

        # Create target TimeSeries
        target_series = []
        for var in self.target_vars:
            target_series.append(TimeSeries.from_series(train[var]))

        # Stack target variables
        train_target_ts = target_series[0]
        for series in target_series[1:]:
            train_target_ts = train_target_ts.stack(series)

        # Create exogenous TimeSeries
        exog_series = []
        for var in self.exog_vars:
            exog_series.append(TimeSeries.from_series(train[var]))

        # Stack exogenous variables
        train_exog_ts = exog_series[0]
        for series in exog_series[1:]:
            train_exog_ts = train_exog_ts.stack(series)

        return train_target_ts, train_exog_ts

    def create_model(self, forecast_horizon: int) -> TiDEModel:
        # Keep minimal HPs + force CPU for reproducibility
        return TiDEModel(
            input_chunk_length=6,
            output_chunk_length=forecast_horizon,
            n_epochs=300,
            random_state=0,
            pl_trainer_kwargs={"accelerator": "cpu", "devices": 1}
        )

    def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
        forecasts = {}

        for horizon in forecast_horizons:
            print(f"\nGenerating {horizon}-month forecast...")

            # Prepare data
            train_target_ts, train_exog_ts = self.prepare_data(horizon)

            # Create and train model
            model = self.create_model(horizon)
            print(f"Training model for {horizon}-month horizon...")
            model.fit(
                series=train_target_ts,
                past_covariates=train_exog_ts,
                verbose=True
            )

            # Generate forecast
            print(f"Generating {horizon}-month predictions...")
            pred = model.predict(
                n=horizon,
                series=train_target_ts,
                past_covariates=train_exog_ts
            )

            # Convert to DataFrame
            pred_df = pd.DataFrame({
                "forecast_inflation": pred.pd_dataframe().iloc[:, 0],
                "forecast_unemployment": pred.pd_dataframe().iloc[:, 1],
                "forecast_oil_price": pred.pd_dataframe().iloc[:, 2],
                "forecast_eer": pred.pd_dataframe().iloc[:, 3],
                "forecast_ir": pred.pd_dataframe().iloc[:, 4]
            })

            forecasts[horizon] = pred_df

            # Save forecast to CSV
            output_file = f"japan_forecasts_tidex_{horizon}m.csv"
            pred_df.to_csv(output_file, index=True)
            print(f"Forecast saved to {output_file}")

        return forecasts


def main():
    generator = japanForecastGenerator(
        data_path="/content/all_mulvar_data_japan_v2.csv",
        random_seed=1
    )

    generator.generate_forecasts([12, 24])

    print("\nCreated files:")
    for horizon in [12, 24]:
        print(f"japan_forecasts_tidex_{horizon}m.csv")


if __name__ == "__main__":
    main()

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/setup.py:187: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type             | Params
---------------------------------------------------------
0 | criterion           | MSELoss          | 0     
1 | train_criterion     | MSELoss          | 0     
2 | val_criterion       | MSELoss          | 0     
3 | train_metrics       | MetricCollection | 0     
4 | val_metrics         | MetricCollection | 0     
5 | past_cov_projection | _ResidualBlock   | 1.2 K 
6 | encoders            | Sequential       


Generating 12-month forecast...
Training model for 12-month horizon...


Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Generating 12-month predictions...


...

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Generating 24-month predictions...


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast saved to japan_forecasts_tidex_24m.csv

Created files:
japan_forecasts_tidex_12m.csv
japan_forecasts_tidex_24m.csv


**Forecasts**
 - **Country: UK**
 - **Forecast Horizons:12M and 24M Forecast**

In [8]:
import random
import pandas as pd
import numpy as np
import torch

from darts import TimeSeries, concatenate
from darts.dataprocessing.transformers import Scaler
from darts.models import NBEATSModel, TFTModel, TiDEModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict


class ukForecastGenerator:
    def __init__(self, data_path: str, random_seed: int = 1):
        # Seeds for reproducibility
        random.seed(random_seed)
        torch.manual_seed(random_seed)
        np.random.seed(random_seed)
        torch.use_deterministic_algorithms(True)

        # Load data
        self.data = pd.read_csv(data_path)

        # Define target and exogenous variables
        self.target_vars = [
            "CPIinflationrate",
            "Unemploymentrate",
            "OilpriceGlobalWTI",
            "RealbroadEER",
            "ShorttermIR"
        ]

        self.exog_vars = [
            "logEPU",
            "GPRC",
            "USEMV",
            "USMPU"
        ]

    def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
        # Split data based on forecast horizon
        train = self.data.head(-forecast_horizon).copy()

        # Create target TimeSeries
        target_series = []
        for var in self.target_vars:
            target_series.append(TimeSeries.from_series(train[var]))

        # Stack target variables
        train_target_ts = target_series[0]
        for series in target_series[1:]:
            train_target_ts = train_target_ts.stack(series)

        # Create exogenous TimeSeries
        exog_series = []
        for var in self.exog_vars:
            exog_series.append(TimeSeries.from_series(train[var]))

        # Stack exogenous variables
        train_exog_ts = exog_series[0]
        for series in exog_series[1:]:
            train_exog_ts = train_exog_ts.stack(series)

        return train_target_ts, train_exog_ts

    def create_model(self, forecast_horizon: int) -> TiDEModel:
        # Keep minimal HPs + force CPU for reproducibility
        return TiDEModel(
            input_chunk_length=6,
            output_chunk_length=forecast_horizon,
            n_epochs=300,
            random_state=0,
            pl_trainer_kwargs={"accelerator": "cpu", "devices": 1}
        )

    def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
        forecasts = {}

        for horizon in forecast_horizons:
            print(f"\nGenerating {horizon}-month forecast...")

            # Prepare data
            train_target_ts, train_exog_ts = self.prepare_data(horizon)

            # Create and train model
            model = self.create_model(horizon)
            print(f"Training model for {horizon}-month horizon...")
            model.fit(
                series=train_target_ts,
                past_covariates=train_exog_ts,
                verbose=True
            )

            # Generate forecast
            print(f"Generating {horizon}-month predictions...")
            pred = model.predict(
                n=horizon,
                series=train_target_ts,
                past_covariates=train_exog_ts
            )

            # Convert to DataFrame
            pred_df = pd.DataFrame({
                "forecast_inflation": pred.pd_dataframe().iloc[:, 0],
                "forecast_unemployment": pred.pd_dataframe().iloc[:, 1],
                "forecast_oil_price": pred.pd_dataframe().iloc[:, 2],
                "forecast_eer": pred.pd_dataframe().iloc[:, 3],
                "forecast_ir": pred.pd_dataframe().iloc[:, 4]
            })

            forecasts[horizon] = pred_df

            # Save forecast to CSV
            output_file = f"uk_forecasts_tidex_{horizon}m.csv"
            pred_df.to_csv(output_file, index=True)
            print(f"Forecast saved to {output_file}")

        return forecasts


def main():
    generator = ukForecastGenerator(
        data_path="/content/all_mulvar_data_uk_v2.csv",
        random_seed=1
    )

    generator.generate_forecasts([12, 24])

    print("\nCreated files:")
    for horizon in [12, 24]:
        print(f"uk_forecasts_tidex_{horizon}m.csv")


if __name__ == "__main__":
    main()

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/setup.py:187: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type             | Params
---------------------------------------------------------
0 | criterion           | MSELoss          | 0     
1 | train_criterion     | MSELoss          | 0     
2 | val_criterion       | MSELoss          | 0     
3 | train_metrics       | MetricCollection | 0     
4 | val_metrics         | MetricCollection | 0     
5 | past_cov_projection | _ResidualBlock   | 1.2 K 
6 | encoders            | Sequential       


Generating 12-month forecast...
Training model for 12-month horizon...


Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Generating 12-month predictions...


...

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Generating 24-month predictions...


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast saved to uk_forecasts_tidex_24m.csv

Created files:
uk_forecasts_tidex_12m.csv
uk_forecasts_tidex_24m.csv


**Forecasts**
 - **Country: Italy**
 - **Forecast Horizons:12M and 24M Forecast**

In [9]:
import random
import pandas as pd
import numpy as np
import torch

from darts import TimeSeries, concatenate
from darts.dataprocessing.transformers import Scaler
from darts.models import NBEATSModel, TFTModel, TiDEModel
from darts.metrics import mape, rmse
from typing import List, Tuple, Dict


class italyForecastGenerator:
    def __init__(self, data_path: str, random_seed: int = 1):
        # Seeds for reproducibility
        random.seed(random_seed)
        torch.manual_seed(random_seed)
        np.random.seed(random_seed)
        torch.use_deterministic_algorithms(True)

        # Load data
        self.data = pd.read_csv(data_path)

        # Define target and exogenous variables
        self.target_vars = [
            "CPIinflationrate",
            "Unemploymentrate",
            "OilpriceGlobalWTI",
            "RealbroadEER",
            "ShorttermIR"
        ]

        self.exog_vars = [
            "logEPU",
            "GPRC",
            "USEMV",
            "USMPU"
        ]

    def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
        # Split data based on forecast horizon
        train = self.data.head(-forecast_horizon).copy()

        # Create target TimeSeries
        target_series = []
        for var in self.target_vars:
            target_series.append(TimeSeries.from_series(train[var]))

        # Stack target variables
        train_target_ts = target_series[0]
        for series in target_series[1:]:
            train_target_ts = train_target_ts.stack(series)

        # Create exogenous TimeSeries
        exog_series = []
        for var in self.exog_vars:
            exog_series.append(TimeSeries.from_series(train[var]))

        # Stack exogenous variables
        train_exog_ts = exog_series[0]
        for series in exog_series[1:]:
            train_exog_ts = train_exog_ts.stack(series)

        return train_target_ts, train_exog_ts

    def create_model(self, forecast_horizon: int) -> TiDEModel:
        # Keep minimal HPs + force CPU for reproducibility
        return TiDEModel(
            input_chunk_length=6,
            output_chunk_length=forecast_horizon,
            n_epochs=300,
            random_state=0,
            pl_trainer_kwargs={"accelerator": "cpu", "devices": 1}
        )

    def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
        forecasts = {}

        for horizon in forecast_horizons:
            print(f"\nGenerating {horizon}-month forecast...")

            # Prepare data
            train_target_ts, train_exog_ts = self.prepare_data(horizon)

            # Create and train model
            model = self.create_model(horizon)
            print(f"Training model for {horizon}-month horizon...")
            model.fit(
                series=train_target_ts,
                past_covariates=train_exog_ts,
                verbose=True
            )

            # Generate forecast
            print(f"Generating {horizon}-month predictions...")
            pred = model.predict(
                n=horizon,
                series=train_target_ts,
                past_covariates=train_exog_ts
            )

            # Convert to DataFrame
            pred_df = pd.DataFrame({
                "forecast_inflation": pred.pd_dataframe().iloc[:, 0],
                "forecast_unemployment": pred.pd_dataframe().iloc[:, 1],
                "forecast_oil_price": pred.pd_dataframe().iloc[:, 2],
                "forecast_eer": pred.pd_dataframe().iloc[:, 3],
                "forecast_ir": pred.pd_dataframe().iloc[:, 4]
            })

            forecasts[horizon] = pred_df

            # Save forecast to CSV
            output_file = f"italy_forecasts_tidex_{horizon}m.csv"
            pred_df.to_csv(output_file, index=True)
            print(f"Forecast saved to {output_file}")

        return forecasts


def main():
    generator = italyForecastGenerator(
        data_path="/content/all_mulvar_data_italy_v2.csv",
        random_seed=1
    )

    generator.generate_forecasts([12, 24])

    print("\nCreated files:")
    for horizon in [12, 24]:
        print(f"italy_forecasts_tidex_{horizon}m.csv")


if __name__ == "__main__":
    main()

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/setup.py:187: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type             | Params
---------------------------------------------------------
0 | criterion           | MSELoss          | 0     
1 | train_criterion     | MSELoss          | 0     
2 | val_criterion       | MSELoss          | 0     
3 | train_metrics       | MetricCollection | 0     
4 | val_metrics         | MetricCollection | 0     
5 | past_cov_projection | _ResidualBlock   | 1.2 K 
6 | encoders            | Sequential       


Generating 12-month forecast...
Training model for 12-month horizon...


Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Generating 12-month predictions...


...

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Generating 24-month predictions...


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast saved to italy_forecasts_tidex_24m.csv

Created files:
italy_forecasts_tidex_12m.csv
italy_forecasts_tidex_24m.csv
